# Credit Risk prediction - Proof of Concept
**Goal** Predict whether a loan will default (Charged Off)
**Data:** LendingClub loan-level data(`loans.csv`). This notebook samples 5000 loans for speed; the approach generalizes to the full dataset.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression 
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix

df = pd.read_csv("loans.csv")
print(df.shape)

/var/folders/j7/ngv7q92n4t32bb8j10gtnjw80000gn/T/ipykernel_73770/2510914344.py:7: DtypeWarning: Columns (0: desc, 1: next_pymnt_d, 2: verification_status_joint, 3: sec_app_earliest_cr_line, 4: hardship_type, 5: hardship_reason, 6: hardship_status, 7: hardship_start_date, 8: hardship_end_date, 9: payment_plan_start_date, 10: hardship_loan_status, 11: debt_settlement_flag_date, 12: settlement_status, 13: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("loans.csv")


(2260668, 145)


## Data Cleaning & Target Definition
We select a random 5,000-loan sample (not the first N rows - the first rows are all recent, unresolved loans, discovered on Day 10). We define`is_default` as 1 for Charged Off, 0 for Fully Paid, and exculde loans still in progress (Current,Late,In Grace Period) sincetheir outcome isn't known yet.

In [3]:
cols = ["loan_amnt","int_rate", "grade","term","annual_inc","loan_status"]
core=df[cols].sample(n=5000, random_state=42)

resolved=core[core["loan_status"].isin(["Fully Paid","Charged Off"])].copy()
resolved["is_default"] = (resolved["loan_status"] =="Charged Off").astype(int)

grade_order = ["A","B","C","D","E","F","G"]
resolved["grade_num"]=resolved["grade"].astype(pd.CategoricalDtype(categories=grade_order,ordered=True)).cat.codes
resolved["term_months"]=resolved["term"].str.replace("months","",regex=False).astype(int)

print(resolved.shape)
print(resolved["is_default"].value_counts())

(2904, 9)
is_default
0    2345
1     559
Name: count, dtype: int64


## Features, Target, and Train/Test Split
**Features used:** `loan_amnt`, `int_rate`, `annual_inc`, `term_months` - all known at loan issuance.

**Excluded:** post-outcome columns like `total_rec_prncp` and `recoveries` (Day 11- these leak the answer, since they're only populated after a loan resolved), and `grade` (Day 15 - 0.95 correlated with `int_rate`, so it's redundant rather than adding independent signal).

Split is 80/20, stratified on the target so both sets have a matching default rate.


In [4]:
features = resolved[["loan_amnt","int_rate", "annual_inc", "term_months"]]
target =resolved["is_default"]

X_train,X_test,y_train,y_test = train_test_split(
    features,target,test_size=0.2, random_state=42, stratify=target
)
print(X_train.shape, X_test.shape)
print("Train default rate:", y_train.mean(), "| Test default rate:", y_test.mean())

(2323, 4) (581, 4)
Train default rate: 0.19242359018510546 | Test default rate: 0.1927710843373494


## Model Training & Evaluation

We train logistic regression on standarized features (scale fit on training data only, to avoid leaking test-set statistics - Day 15).

** Why not just accuracy:** with only ~19% of loans defaulting, a model that never predicts default scores ~81% accuracy while catching zero real defaults. We evaluate with AUC (ranking quality, independent of any cutoff) and recall/precision at multiple thresholds (Day 13-14).


In [5]:
scaler=StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:,1]
baseline_auc = 0.5
model_auc = roc_auc_score(y_test, y_proba)

print("Model AUC:", model_auc, "| Baseline AUC:", baseline_auc)
for name, coef in sorted(zip(features.columns, model.coef_[0]), key=lambda x: abs(x[1]), reverse=True):
    print(name,":",coef)

Model AUC: 0.7261936491014316 | Baseline AUC: 0.5
int_rate : 0.557052683059718
annual_inc : -0.1056767448013292
loan_amnt : 0.08161876419245093
term_months : 0.04913315534075361


## Threshold Tuning

The default 0.5 probability cutoff is arbitrary, not something the model chose for a reason. We test several thresholds to see the accuracy/precision/recall trade-off, since in lending, missing a real default is usually costlier than flagging a safe loan for extra review (Day 14).


In [6]:
for threshold in [0.5,0.4,0.3,0.15]:
    y_pred_thresh = (y_proba >= threshold).astype(int)
    precision = precision_score(y_test, y_pred_thresh, zero_division=0)
    recall = recall_score(y_test, y_pred_thresh, zero_division=0)
    accuracy = (y_pred_thresh == y_test).mean()
    print(f"Threshold {threshold}: accuracy={accuracy:.3f}, precision={precision:.3f}, recall={recall:.3f}")

Threshold 0.5: accuracy=0.809, precision=0.600, recall=0.027
Threshold 0.4: accuracy=0.816, precision=0.609, recall=0.125
Threshold 0.3: accuracy=0.787, precision=0.400, recall=0.214
Threshold 0.15: accuracy=0.573, precision=0.285, recall=0.804


In [7]:
chosen_threshold =0.4
y_pred_final = (y_proba >= chosen_threshold).astype(int)
print(confusion_matrix(y_test, y_pred_final))

[[460   9]
 [ 98  14]]


## Summary: Strengths and Limitations

**What this model does well:**
- Meaningfully beats random guessing: AUC of 0.73 vs. a baseline of 0.5, meaning it genuinely ranks risky loans higher than safe ones, not just by chance.
- Uses only features known at loan issuance (no leakage from post-outcome data like `total_rec_prncp` or `recoveries`).
- Avoids a redundant feature: `grade` was dropped after confirming 0.95 correlation with `int_rate` — removing it cost almost no predictive power (AUC dropped only 0.003) while making the model more interpretable.
- Threshold tuning (0.4 instead of the default 0.5) improves accuracy, precision, *and* recall simultaneously versus a naive cutoff — a free win, not just a trade-off.

**Limitations:**
- **Recall is still low.** Even after tuning, the model only catches 12.5% of actual defaults at threshold 0.4 (14 of 112 in the test set). Catching more requires accepting far more false alarms (recall reaches ~80% only at threshold 0.15, where precision drops to 28%). The "right" threshold depends on the real dollar cost of a missed default vs. a false alarm — a business input this dataset doesn't provide.
- **Small sample.** This model trains on 5,000 sampled loans (2,904 resolved); the full dataset has 2.26 million rows. Results may not hold at full scale.
- **Random, not chronological, split.** Train/test was split randomly, not by time. A model deployed in production would be tested on genuinely *future* loans, where economic conditions may differ from the training period — this notebook doesn't test that.
- **Few features.** Only 4 features are used (`loan_amnt`, `int_rate`, `annual_inc`, `term_months`). A production underwriting model would likely use dozens, including credit history and behavioral data.
- **Single model type.** Only logistic regression was tried. Day 17 will compare against a random forest to see if a more flexible model captures more signal.

## Alternative Model: Random Forest

Logistic regression assumes linear relationships between features and risk. Random forest can capture non-linear patterns and feature interactions automatically. We compare it honestly against the logistic regression above, on the identical train/test split.

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf_model.fit(X_train,y_train)

rf_proba = rf_model.predict_proba(X_test)[:,1]
rf_auc = roc_auc_score(y_test,rf_proba)

print("Random Forest AUC:",rf_auc)
print("Logistic Regression AUC:",model_auc)

Random Forest AUC: 0.7157801553457204
Logistic Regression AUC: 0.7261936491014316


In [10]:
rf_importances = sorted(zip(features.columns,rf_model.feature_importances_),key=lambda x:x[1],reverse=True)
for name,importance in rf_importances:
    print(name,":",importance)

int_rate : 0.5498793321029273
annual_inc : 0.20784694169803464
loan_amnt : 0.19431454534032258
term_months : 0.04795918085871554


## Model Comparison Verdict

Logistic regression (AUC 0.726) slightly outperformed random forest (AUC 0.716) on this dataset — a reminder that more complex models aren't automatically better, especially with few features and a modest sample size. Both models independently agree on feature importance ranking (int_rate > annual_inc > loan_amnt > term_months), which strengthens confidence in the finding.

**Choosing logistic regression as the final model**, for two reasons: (1) it performed at least as well, and (2) it's interpretable — lending decisions are subject to fair lending regulations (e.g., ECOA in the US) that often require explaining *why* an applicant was denied credit. A logistic regression coefficient directly answers that; a random forest's prediction is much harder to explain to a regulator or applicant. In a regulated domain, "slightly better AUC" from a black-box model isn't automatically worth the interpretability cost.